In [50]:
import numpy as np

# Read text

In [51]:
with open('../data/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [52]:
print(f'Characters: {len(text)}')

Characters: 1115393


In [53]:
print(f'{text[:100]}')

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [54]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f'All characters: {"".join(chars)}')
print(f'Vocab size: {vocab_size}')

All characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab size: 65


# Character level tokenizer

In [55]:
encoded_dict = {chars[i]: i for i in range(vocab_size)}
decoded_dict = {i: chars[i] for i in range(vocab_size)}

def encode(s: str) -> list[int]:
    return [encoded_dict[c] for c in s]

def decode(s: list[int]) -> str:
    return "".join([decoded_dict[c] for c in s])

print(encode("test string"))
print(decode(encode("test string")))

[58, 43, 57, 58, 1, 57, 58, 56, 47, 52, 45]
test string


In [56]:
data = encode(text)
data = np.array(data, dtype=np.float64).reshape(-1, 1)
print(data.shape)

(1115393, 1)


In [57]:
n = int(0.9*len(data))
X_train, X_test = data[:n], data[n:]
#print(f'n: {n}\nX_train: {X_train.shape}\nX_test: {X_test.shape}')

In [58]:
def create_sequences(data, seq_len = 8):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len, :])
        y.append(data[i+1:i+seq_len+1, :])
    return np.array(X), np.array(y)

X_train_seq, y_train_seq = create_sequences(X_train)
print(f'{X_train_seq.shape}, {y_train_seq.shape}')

(1003845, 8, 1), (1003845, 8, 1)


In [59]:
from dlfs.layers import DenseLayer, DropoutLayer
from dlfs.activation import Softmax
from dlfs.base import Layer

class SingleAttentionHead():

    def __init__(self, n_embed, head_size, block_size, dropout=0.1):
        self.key = DenseLayer(n_embed, head_size)
        self.query = DenseLayer(n_embed, head_size)
        self.value = DenseLayer(n_embed, head_size)
        self.softmax = Softmax()
        self.dropout = DropoutLayer(dropout)

        self.tril = np.tril(np.ones((block_size, block_size)))
        self.normalize_factor = head_size**0.5

    def forward(self, x, training):
        B, T, C = x.shape

        self.key.forward(x)
        self.query.forward(x)
        self.value.forward(x)

        self.k = self.key.output
        self.q = self.query.output
        self.v = self.value.output
        
        self.w = np.matmul(self.q, self.k.swapaxes(-2, -1)) / self.normalize_factor
        mask_condition = self.tril[:T, :T] == 0
        self.w[:,mask_condition] = -np.inf
        self.softmax.forward(self.w)
        self.w = self.softmax.output
        self.dropout.forward(self.w, training)
        self.w = self.dropout.output
        self.output = np.matmul(self.w, self.v)

    def backward(self, delta):

        # Step 1: Gradient of the loss with respect to w (attention weights)
        d_w = np.matmul(delta, self.v.swapaxes(-2, -1))

        self.dropout.backward(d_w)  # This will apply the dropout mask to the gradients
        d_w = self.dropout.dinputs

        # Step 2: Gradient of the loss with respect to softmax input (logits)
        self.softmax.backward(d_w)  # Softmax backward pass
        d_w = self.softmax.dinputs

        # Step 3: Gradient of the loss with respect to w (before softmax)
        d_w = d_w * (self.w > 0).astype(float)  # Masking out invalid values from softmax

        # Step 4: Gradients w.r.t. key and query using the chain rule
        d_q = np.matmul(d_w, self.k)  # shape: (B, T, head_size)
        d_k = np.matmul(d_w.swapaxes(-2, -1), self.q)  # shape: (B, T, head_size)

        # Step 5: Update the key, query, and value parameters using the gradients
        # Gradient for the key (d_k) and query (d_q) go through the dense layers
        self.key.backward(d_k)
        self.query.backward(d_q)
        self.value.backward(np.matmul(d_w, self.v))

        self.dinputs = self.key.dinputs + self.query.dinputs + self.value.dinputs

In [60]:
embedding = DenseLayer(1, 192)
head = SingleAttentionHead(n_embed=192, head_size=56, block_size=8)

x = X_train_seq[0]
x = x.reshape(1, *x.shape)
print(f'x {x.shape}')
embedding.forward(x, training=True)
print(f'embedding: {embedding.output.shape}')
head.forward(embedding.output, training=True)
print(f'output shape: {head.output.shape}')

x (1, 8, 1)
embedding: (1, 8, 192)
output shape: (1, 8, 56)


In [61]:
delta = np.random.rand(1, 8, 56)
head.backward(delta)
print(head.key.dinputs.shape, head.query.dinputs.shape, head.value.dinputs.shape)

(1, 8, 192) (1, 8, 192) (1, 8, 192)


In [62]:
class MultiHeadAttention():

    def __init__(self, n_embed, n_heads, head_size, block_size, dropout=0.1):
        self.n_heads = n_heads
        self.head_size = head_size

        # List to store each individual attention head
        self.attention_heads = [
            SingleAttentionHead(n_embed, head_size, block_size, dropout)
            for _ in range(n_heads)
        ]

        # Output Dense layer to combine the heads
        self.output_dense = DenseLayer(n_embed, n_embed)

        self.dropout = DropoutLayer(dropout)

    def forward(self, x, training):

        # Store outputs of all attention heads
        head_outputs = []

        for head in self.attention_heads:
            head.forward(x, training)  # Compute attention for this head
            head_outputs.append(head.output)  # Store the output of each head

        # Concatenate the outputs of all heads along the last dimension (features)
        concatenated_output = np.concatenate(np.array(head_outputs), axis=-1) 

        # Pass the concatenated output through the output dense layer
        self.output_dense.forward(concatenated_output)

        self.dropout.forward(self.output_dense.output, training)

        # Final output
        self.output = self.dropout.output

    def backward(self, delta):

        self.dropout.backward(delta)

        self.output_dense.backward(self.dropout.dinputs)

        d_concatenated_output = self.output_dense.output

        # Step 2: Split the gradient back into the individual heads
        d_head_outputs = np.split(d_concatenated_output, self.n_heads, axis=-1)

        # Step 3: Backpropagate through each attention head
        for i, head in enumerate(self.attention_heads):
            head.backward(d_head_outputs[i])  # Backprop through each head

        self.dinputs = self.attention_heads[0].dinputs

In [63]:
n_embed = 192
n_heads = 8
head_size = n_embed // n_heads

multihead = MultiHeadAttention(n_embed=n_embed, n_heads=n_heads, head_size=head_size, block_size=8)

embedding.forward(x)
multihead.forward(embedding.output, training=True)
print(f'x: {x.shape}')
print(f'embedding: {embedding.output.shape}')
print(f'multihead: {multihead.output.shape}')

x: (1, 8, 1)
embedding: (1, 8, 192)
multihead: (1, 8, 192)


In [64]:
delta = np.random.rand(1, 8, 192)
multihead.backward(delta)
for idx, head in enumerate(multihead.attention_heads):
    print(idx, head.key.dinputs.shape, head.query.dinputs.shape, head.value.dinputs.shape)

0 (1, 8, 192) (1, 8, 192) (1, 8, 192)
1 (1, 8, 192) (1, 8, 192) (1, 8, 192)
2 (1, 8, 192) (1, 8, 192) (1, 8, 192)
3 (1, 8, 192) (1, 8, 192) (1, 8, 192)
4 (1, 8, 192) (1, 8, 192) (1, 8, 192)
5 (1, 8, 192) (1, 8, 192) (1, 8, 192)
6 (1, 8, 192) (1, 8, 192) (1, 8, 192)
7 (1, 8, 192) (1, 8, 192) (1, 8, 192)


In [65]:
from dlfs.activation import ReLU

class FeedForward():

    def __init__(self, n_embed, dropout=0.1):
        self.fc1 = DenseLayer(n_embed, 4*n_embed)
        self.relu1 = ReLU()
        self.fc2 = DenseLayer(4*n_embed, n_embed)
        self.relu2 = ReLU()
        self.dropout = DropoutLayer(dropout)

    def forward(self, inputs, training):
        self.fc1.forward(inputs)
        self.relu1.forward(self.fc1.output)
        self.fc2.forward(self.relu1.output)
        self.relu2.forward(self.fc2.output)
        self.dropout.forward(self.relu2.output, training)
        self.output = self.dropout.output

    def backward(self, delta):
        self.dropout.backward(delta)
        self.relu2.backward(self.dropout.dinputs)
        self.fc2.backward(self.relu2.dinputs)
        self.relu1.backward(self.fc2.dinputs)
        self.fc1.backward(self.relu1.dinputs)
        self.dinputs = self.fc1.dinputs

In [66]:
class LayerNorm(Layer):
    def __init__(self, num_features, epsilon=1e-5):
        """
        Initializes the LayerNorm layer.
        
        :param num_features: The number of features in the input (i.e., the dimension to normalize over).
        :param epsilon: Small value to prevent division by zero when computing the standard deviation.
        """
        self.num_features = num_features
        self.epsilon = epsilon
        
        # Initialize the scale (gamma) and shift (beta) parameters
        self.gamma = np.ones(num_features)  # Shape: num_features
        self.beta = np.zeros(num_features)  # Shape: (1, num_features)
        
    def forward(self, inputs, training=False):
        """
        Forward pass of LayerNorm
        
        :param x: Input data of shape (batch_size, num_features)
        :return: Layer normalized output
        """
        mean = np.mean(inputs, axis=-1, keepdims=True)
        variance = np.var(inputs, axis=-1, keepdims=True)

        self.normalized = (inputs - mean) / np.sqrt(variance + self.epsilon)
        self.output = self.gamma * self.normalized + self.beta
    
    def backward(self, delta):
        """
        Backward pass for LayerNorm, computing the gradients.
        
        :param dout: The gradient of the loss with respect to the output.
        :return: Gradients with respect to input (dx), gamma, and beta.
        """
        self.dbeta = np.sum(delta, axis=(0, 1))
        self.dgamma = np.sum(delta * self.normalized, axis=(0, 1))
        dnorm = delta * self.gamma
        self.dinputs = dnorm - np.mean(dnorm, axis=-1, keepdims=True) - self.normalized * np.mean(dnorm * self.normalized, axis=-1, keepdims=True)

In [67]:
np.random.seed(21)

data = np.random.rand(1, 8, 15)
ln = LayerNorm(15)
ln.forward(data, training=True)
print(ln.output.shape)

(1, 8, 15)


In [68]:
delta = np.random.rand(1, 8, 15)
ln.backward(delta)
print(ln.dgamma.shape, ln.dbeta.shape, ln.dinputs.shape)

(15,) (15,) (1, 8, 15)


In [69]:
class Block:
    def __init__(self, n_embed, n_head, block_size, dropout=0.1):
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_heads=n_head, head_size=head_size, n_embed=n_embed, block_size=block_size, dropout=dropout)
        self.ffwd = FeedForward(n_embed, dropout)
        self.ln1 = LayerNorm(n_embed)
        self.ln2 = LayerNorm(n_embed)
    def forward(self, x, training):
        self.ln1.forward(x, training)
        self.sa.forward(self.ln1.output, training)
        x = x + self.sa.output
        self.ln2.forward(x, training)
        self.ffwd.forward(self.ln2.output, training)
        x = x + self.ffwd.output
        self.output = x

    def backward(self, delta):

        dx = delta
        dffwd = dx  # Gradient to pass to the FeedForward layer
        
        self.ffwd.backward(dffwd)

        self.ln2.backward(dx)
        dln2 = self.ln2.dinputs
        
        dsa = dln2  # Gradient to pass to MultiHeadAttention
        
        self.sa.backward(dsa)
        
        self.ln1.backward(dsa)

        self.dinputs = self.ln1.dinputs

In [70]:
n_embed = 192
n_heads = 8

b = Block(n_embed, n_heads, 8)

x = X_train_seq[0]
x = x.reshape(1, *x.shape)
embedding.forward(x)
b.forward(embedding.output, training=True)
print(f'x: {x.shape}')
print(f'embedding: {embedding.output.shape}')
print(f'block: {b.output.shape}')

x: (1, 8, 1)
embedding: (1, 8, 192)
block: (1, 8, 192)


In [71]:
delta = np.random.rand(1, 8, 192)
b.backward(delta)
print(b.dinputs.shape)
embedding.backward(b.dinputs)
print(embedding.dinputs.shape)

(1, 8, 192)
(1, 8, 1)


In [72]:
class EmbeddingLayer(Layer):

    def __init__(self, vocab_size, embedding_dim):
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.embeddings = np.random.randn(vocab_size, embedding_dim) * 0.01  # Initialize with small random values

    def forward(self, input_indices, training):
        """
        Forward pass: Given input indices, retrieve corresponding embeddings.
        """
        self.input_indices = input_indices
        self.output = self.embeddings[input_indices]

    def backward(self, delta):

        self.dembedding = []

        for i, idx in enumerate(self.input_indices):
            # The gradient w.r.t. the embedding is simply the gradient w.r.t. the output
            # (d_output[i]) because of the identity mapping in the embedding lookup
            self.dembedding.append(delta[i])

        self.dembedding = np.array(self.dembedding)

In [73]:
class PositionalEncoding(Layer):

    def __init__(self, sequence_length, n_embed):
        self.sequence_length = sequence_length
        self.n_embed = n_embed

    def _positional_encode(self):
        P = np.zeros((self.sequence_length, self.n_embed))
        for k in range(self.sequence_length):
            for i in np.arange(int(self.n_embed/2)):
                denominator = 10000**(2*i / self.n_embed)
                P[k, 2*i] = np.sin(k/denominator)
                P[k, 2*i+1] = np.cos(k/denominator)
        return P

    def forward(self, inputs, training):
        self.output = inputs + self._positional_encode()

    def backward(self, delta):
        self.dinputs = delta


In [74]:
emb = EmbeddingLayer(vocab_size, 192)
pos = PositionalEncoding(8, 192)
x = X_train_seq[:2].astype(int).squeeze(axis=-1)
print(x.shape)
emb.forward(x, training=True)
pos.forward(emb.output, training=True)

print(emb.output.shape)
print(pos.output.shape)

(2, 8)
(2, 8, 192)
(2, 8, 192)


In [75]:
class TransformerDecoder:

    def __init__(self, n_embed, n_head, block_size, n_layers: int = 1, dropout=0.1) -> None:
        if n_layers == 1:
            self.blocks = [Block(n_embed, n_head, block_size, dropout)]
        else:
            self.blocks = [Block(n_embed, n_head, block_size, dropout) for _ in range(n_layers)]

    def forward(self, inputs: np.ndarray, training) -> None:

        # Pass data to the first LSTM layer
        self.blocks[0].forward(inputs, training)

        # Forward hidden states of the previous LSTM layer to the current one
        for idx, layer in enumerate(self.blocks[1:], start=1):
            layer.forward(self.blocks[idx - 1].output, training)

        # Output of the LSTM is the final LSTM layer's output
        self.output = self.blocks[-1].output.copy()

    def backward(self, delta: np.ndarray) -> None:

        self.blocks[-1].backward(delta)

        for idx, layer in reversed(list(enumerate(self.blocks[:-1]))):
            layer.backward(self.blocks[idx + 1].dinputs)

        self.dinputs = self.blocks[0].dinputs

In [76]:
from dlfs.layers import EmbeddingLayer, PositionalEncoding, LayerNorm, DenseLayer
from dlfs.modules import TransformerDecoder
from dlfs.activation import Softmax
from dlfs.base import Module

def get_random_batch(X, y, block_size, batch_size):
    idx = np.random.randint(0, len(X) - block_size, size=(batch_size, ))
    return X[idx], y[idx]

class TransformerDecoderModel(Module):

    def __init__(self, vocab_size, block_size, n_embed, n_layers, n_head, dropout, loss_function, optimizer):
        self.block_size = block_size
        self.loss_function = loss_function
        self.optimizer = optimizer
        self.layers = [EmbeddingLayer(vocab_size, n_embed),
                       PositionalEncoding(block_size, n_embed),
                       TransformerDecoder(n_embed, n_head, block_size, n_layers, dropout),
                       LayerNorm(n_embed),
                       DenseLayer(n_embed, vocab_size),
                       Softmax()
                       ]

    def forward(self, inputs, training):
        self.layers[0].forward(inputs, training)

        for idx, layer in enumerate(self.layers[1:], start=1):
            layer.forward(self.layers[idx - 1].output, training)

        self.output = self.layers[-1].output

    def backward(self, y):
        output = self.output
        B, T, C = output.shape
        output = output.reshape(B*T, C)
        y = y.reshape(B*T)

        self.loss_function.backward(output, y)
        self.layers[-1].backward(self.loss_function.dinputs.reshape(B, T, C))

        for idx, layer in reversed(list(enumerate(self.layers[:-1]))):
            layer.backward(self.layers[idx + 1].dinputs)

    def train(self, X, y, epochs = 1000, batch_size: int = None, print_every: int = None):

        for i in range(epochs + 1):
            batch_X, batch_y = get_random_batch(X, y, self.block_size, batch_size)
            batch_X, batch_y = batch_X.astype(int).squeeze(-1), batch_y.astype(int).squeeze(-1)

            self.forward(batch_X, training=True)
            
            self.backward(batch_y)

            self.optimizer.pre_update_parameters()
            self.optimizer.update_parameters(self)
            self.optimizer.post_update_parameters()

            if not i % 10:
                output = self.output
                B, T, C = output.shape
                output = output.reshape(B*T, C)
                batch_y = batch_y.reshape(B*T)
                print(f'===== EPOCH : {i} ===== LOSS : {self.loss_function.calculate(output, batch_y)} =====')

    def predict(self, X):
        self.forward(X, training=False)

In [77]:
from dlfs.loss import CCE_Loss
from dlfs.optimizers import Optimizer_Adam

vocab_size = len(chars)
batch_size = 64
block_size = 256
epochs = 5000
lr = 3e-4
n_embed = 192
n_head = 6
n_layers = 3
dropout = 0.2

loss = CCE_Loss()
optimizer = Optimizer_Adam(learning_rate=lr)

model = TransformerDecoderModel(vocab_size, block_size, n_embed, n_layers, n_head, dropout, loss_function=loss, optimizer=optimizer)
model.train(X_train_seq, y_train_seq, print_every=10, epochs=epochs, batch_size=batch_size)

===== EPOCH : 0 ===== LOSS : 5.129886395337548 =====
===== EPOCH : 10 ===== LOSS : 3.8039986944144957 =====


KeyboardInterrupt: 

In [ ]:
def generate(idx, max_new_tokens):
        
        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]

            model.predict(idx_cond)
    
            logits = model.output

            probs = logits[:, -1, :].reshape(-1)

            #print(probs)

            idx_next = np.argmax(np.random.multinomial(n=1, pvals=probs, size=1), axis=1).reshape(1, 1)

            idx = np.concatenate((idx, idx_next), axis=1)

        return idx


In [ ]:
print(decode(generate(idx=np.zeros((1, 1), dtype=np.int16), max_new_tokens=1000)[0].tolist()))


t e eh o
sutge
eoaoye em lr 'l
Tsbsn ehoee; ya shilneoe emr;ef I ssl irr'hc t
a Htto
iu rre   YSd :
 
,wu epsft Ir  Bpt.t,hnn f,evh
s rs
hehtsaehe 
 g o'eMaoyl?tet f  ao n  e ri:  tvf ha
na
 nnE ,e, rwplWath tbb 
s ee  r WuBroenddoeaLg
rhduh rmrsbeotruer
 h   tnEd:de  o a oa
mn hwstnyirtybteeeure on itsh l-u Ihuauynfr d the tiaoduee i d snnrueiWUu iryar
huG h      at
osnnoob R?eao;a hyn riIlnnyeiI
.
Ue 
eh :s,aatdutuio mtianIed
hIteo  o n en eLfHynelmoro eeDTe e ya a
sytr aeuoh o ir
 craas
 n lnauOSbf d
i colrlohr orslIlhrrd,  qto  n ca leyb,eseeeufsd eor
isrme twlYrgdoau'oiuUasf nrwamit  onIiaye taa httlie ,Itgtt  sfocf,hAaEtsatte. eentnsleady w
ea  Cu
h
soaeomth  cft
la
fEw rdl
rBdMr iIl 
eahc anayuepmoteeasenf tiuhm w  se huiedduru l  h ops   erdstkre Te un sC
rueolgaraou RIeyak ttiifu,b ,las eu aetohaAshnoIu  s   tee  usdeeuti nadla  ekhelIdnnne n h'rfeY dps  at: r
  ,n
ewsrIdearau:een,gieosll erfIae
et npsoh
oHaoesphisyaattsotw rr ncSy 
rnBhiEdsarelEui 
e: aatam fdd .teta.oOc  
 

# Full Transformer (Encoder - Decoder)

In [ ]:
class SingleAttentionHead(Module):

    def __init__(self, input_size, head_size, dropout=0.1, use_mask=False):
        self.key = DenseLayer(input_size, head_size)
        self.query = DenseLayer(input_size, head_size)
        self.value = DenseLayer(input_size, head_size)
        self.softmax = Softmax()
        self.dropout = DropoutLayer(dropout)

        self.normalize_factor = head_size**0.5
        self.use_mask = use_mask

    def forward(self, query_input, context_input, training):

        self.query.forward(query_input)
        self.key.forward(context_input)
        self.value.forward(context_input)

        self.q = self.query.output
        self.k = self.key.output
        self.v = self.value.output
        
        self.w = np.matmul(self.q, self.k.swapaxes(-2, -1)) / self.normalize_factor

        if self.use_mask:
            B, T, _ = self.q.shape
            mask = np.tril(np.ones((T, T), dtype=bool))  # causal mask
            self.w = np.where(mask[None, :, :], self.w, -np.inf)


        self.softmax.forward(self.w)
        self.w = self.softmax.output
        self.attn_weights = self.softmax.output.copy()
        self.dropout.forward(self.w, training)
        self.w = self.dropout.output
        self.output = np.matmul(self.w, self.v)

    def backward(self, delta):
        d_w = np.matmul(delta, self.v.swapaxes(-2, -1))

        self.dropout.backward(d_w)
        d_w = self.dropout.dinputs

        self.softmax.backward(d_w)
        d_w = self.softmax.dinputs

        if self.use_mask:
            B, T, _ = d_w.shape
            mask = np.tril(np.ones((T, T), dtype=bool))
            d_w = d_w * mask[None, :, :]

        d_q = np.matmul(d_w, self.k) / self.normalize_factor
        d_k = np.matmul(d_w.swapaxes(-2, -1), self.q) / self.normalize_factor

        d_v_input = np.matmul(self.attn_weights.transpose(0, 2, 1), delta)

        self.key.backward(d_k)
        self.query.backward(d_q)
        self.value.backward(d_v_input)

        self.dinputs_query = self.query.dinputs
        self.dinputs_context = self.key.dinputs + self.value.dinputs

In [ ]:
class MultiHeadAttention(Module):

    def __init__(self, n_embed, n_heads, dropout=0.1, use_mask=False):
        self.n_heads = n_heads
        self.head_size = n_embed // n_heads

        self.attention_heads = [
            SingleAttentionHead(n_embed, self.head_size, dropout, use_mask)
            for _ in range(n_heads)
        ]

        self.output_dense = DenseLayer(n_embed, n_embed)

        self.dropout = DropoutLayer(dropout)

    def forward(self, x, training, context=None):
        self.is_cross_attention = context is not None
        if context is None:
            context = x

        head_outputs = []
        for i, head in enumerate(self.attention_heads):
            head.forward(x, context, training)
            head_outputs.append(head.output)

        concatenated_output = np.concatenate(head_outputs, axis=-1)
        self.output_dense.forward(concatenated_output)
        self.dropout.forward(self.output_dense.output, training)
        self.output = self.dropout.output

    def backward(self, delta):

        self.dropout.backward(delta)

        self.output_dense.backward(self.dropout.dinputs)

        d_concatenated_output = self.output_dense.dinputs

        d_head_outputs = np.split(d_concatenated_output, self.n_heads, axis=-1)

        self.attention_heads[0].backward(d_head_outputs[0])
        self.dinputs_query = np.zeros_like(self.attention_heads[0].dinputs_query)
        self.dinputs_context = np.zeros_like(self.attention_heads[0].dinputs_context)

        self.dinputs_query += self.attention_heads[0].dinputs_query
        self.dinputs_context += self.attention_heads[0].dinputs_context

        for i, head in enumerate(self.attention_heads[1:], start=1):
            head.backward(d_head_outputs[i])

        self.dinputs_query /= self.n_heads
        self.dinputs_context /= self.n_heads

        if self.is_cross_attention:
            self.dinputs = None
        else:
            self.dinputs = self.dinputs_query + self.dinputs_context

In [ ]:
class FeedForward(Module):
    def __init__(self, d_model, hidden_dim=2048, dropout=0.1):
        self.fc1 = DenseLayer(d_model, hidden_dim)
        self.gelu = GeLU()
        self.dropout1 = DropoutLayer(dropout)
        self.fc2 = DenseLayer(hidden_dim, d_model)
        self.dropout2 = DropoutLayer(dropout)

    def forward(self, inputs, training):
        self.fc1.forward(inputs)
        self.gelu.forward(self.fc1.output)
        self.dropout1.forward(self.gelu.output, training)
        self.fc2.forward(self.dropout1.output)
        self.dropout2.forward(self.fc2.output, training)
        self.output = self.dropout2.output

    def backward(self, delta):
        self.dropout2.backward(delta)
        self.fc2.backward(self.dropout2.dinputs)
        self.dropout1.backward(self.fc2.dinputs)
        self.gelu.backward(self.dropout1.dinputs)
        self.fc1.backward(self.gelu.dinputs)
        self.dinputs = self.fc1.dinputs

In [ ]:
class TransformerDecoderBlock(Module):
    
    def __init__(self, d_model, n_head, dim_ff=2048, dropout=0.1):
        self.mask_mha = MultiHeadAttention(d_model, n_head, dropout=dropout, use_mask=True)
        self.ln1 = LayerNorm(d_model)

        self.cross_mha = MultiHeadAttention(d_model, n_head, dropout=dropout, use_mask=False)
        self.ln2 = LayerNorm(d_model)

        self.ffwd = FeedForward(d_model, hidden_dim=dim_ff, dropout=dropout)
        self.ln3 = LayerNorm(d_model)

    def forward(self, x, enc_output, training):
        self.ln1.forward(x, training)
        self.mask_mha.forward(self.ln1.output, training)
        x = x + self.mask_mha.output

        self.ln2.forward(x, training)
        self.cross_mha.forward(self.ln2.output, training, context=enc_output)
        x = x + self.cross_mha.output

        self.ln3.forward(x, training)
        self.ffwd.forward(self.ln3.output, training)
        x = x + self.ffwd.output

        self.output = x

    def backward(self, delta):
        self.ln3.backward(delta)
        self.ffwd.backward(self.ln3.dinputs)
        d_cross_residual = delta + self.ffwd.dinputs

        self.ln2.backward(d_cross_residual)
        self.cross_mha.backward(self.ln2.dinputs)
        d_mask_residual = self.cross_mha.dinputs_query + d_cross_residual
        self.grad_wrt_encoder_output = self.cross_mha.dinputs_context

        self.ln1.backward(d_mask_residual)
        self.mask_mha.backward(self.ln1.dinputs)
        self.dinputs = self.mask_mha.dinputs + delta

In [ ]:
class TransformerEncoderBlock(Module):
    def __init__(self, d_model, n_head, dim_ff=2048, dropout=0.1):
        self.mha = MultiHeadAttention(d_model, n_head, dropout=dropout, use_mask=False)
        self.ln1 = LayerNorm(d_model)

        self.ffwd = FeedForward(d_model, hidden_dim=dim_ff, dropout=dropout)
        self.ln2 = LayerNorm(d_model)

    def forward(self, x, training):
        self.ln1.forward(x, training)
        self.mha.forward(self.ln1.output, training)
        x = x + self.mha.output

        self.ln2.forward(x, training)
        self.ffwd.forward(self.ln2.output, training)
        x = x + self.ffwd.output

        self.output = x

    def backward(self, delta):
        d_ffn_residual = delta
        self.ffwd.backward(d_ffn_residual)
        self.ln2.backward(self.ffwd.dinputs)
        d_mha_residual = self.ln2.dinputs + d_ffn_residual 

        self.mha.backward(d_mha_residual)
        self.ln1.backward(self.mha.dinputs)
        self.dinputs = self.ln1.dinputs + delta

In [ ]:
class Transformer(Module):

    def __init__(self, vocab_size, block_size, n_embed, n_head, n_layers, dim_ff, dropout, loss_function, optimizer):
        
        self.block_size = block_size
        self.loss_function = loss_function
        self.optimizer = optimizer

        self.embed_enc = EmbeddingLayer(vocab_size, n_embed)
        self.pos_enc_enc = PositionalEncoding(block_size, n_embed)

        self.embed_dec = EmbeddingLayer(vocab_size, n_embed)
        self.pos_enc_dec = PositionalEncoding(block_size, n_embed)

        self.encoder = [TransformerEncoderBlock(n_embed, n_head, dim_ff=dim_ff, dropout=dropout) for _ in range(n_layers)]
        self.decoder = [TransformerDecoderBlock(n_embed, n_head, dim_ff=dim_ff, dropout=dropout) for _ in range(n_layers)]
        self.linear = DenseLayer(n_embed, vocab_size)
        self.softm = Softmax()

    def forward(self, inputs_enc, inputs_dec, training):
        self.embed_enc.forward(inputs_enc, training)
        self.pos_enc_enc.forward(self.embed_enc.output, training)

        # Encoder forward
        self.encoder[0].forward(self.pos_enc_enc.output, training)
        for idx, enc in enumerate(self.encoder[1:], start=1):
            enc.forward(self.encoder[idx-1].output, training)

        self.embed_dec.forward(inputs_dec, training)
        self.pos_enc_dec.forward(self.embed_dec.output, training)

        # Decoder forward
        self.decoder[0].forward(self.pos_enc_dec.output, self.encoder[-1].output, training)
        for idx, dec in enumerate(self.decoder[1:], start=1):
            dec.forward(self.decoder[idx-1].output, self.encoder[-1].output, training)

        self.linear.forward(self.decoder[-1].output)
        self.softm.forward(self.linear.output)
        self.output = self.softm.output

    def backward(self, output, y):
        y_pred = output
        B, T, C = y_pred.shape
        y_pred = y_pred.reshape(B*T, C)
        y_true = y.reshape(B*T)

        self.loss_function.backward(y_pred, y_true)

        #self.softm.backward(self.loss_function.dinputs.reshape(B, T, C))
        self.linear.backward(self.loss_function.dinputs.reshape(B, T, C))

        self.decoder[-1].backward(self.linear.dinputs)
        for idx, dec in reversed(list(enumerate(self.decoder[:-1]))):
            dec.backward(self.decoder[idx + 1].dinputs)

        grad_wrt_encoder_output_total = np.zeros_like(self.decoder[0].output)
        for idx, dec in enumerate(self.decoder):
            grad_wrt_encoder_output_total += dec.grad_wrt_encoder_output

        self.pos_enc_dec.backward(self.decoder[0].dinputs)
        self.embed_dec.backward(self.pos_enc_dec.dinputs)

        self.encoder[-1].backward(grad_wrt_encoder_output_total)
        for idx, enc in reversed(list(enumerate(self.encoder[:-1]))):
            enc.backward(self.encoder[idx + 1].dinputs)

        self.pos_enc_enc.backward(self.encoder[0].dinputs)
        self.embed_enc.backward(self.pos_enc_enc.dinputs)

    def _print_grad_norms(self):
        print(f'-------------------GRADS------------------')
        print(f'CCE: {np.linalg.norm(self.loss_function.dinputs)}')
        print(f'Linear: {np.linalg.norm(self.linear.dweights)}')
        for i in range(len(self.encoder)):
            print(f'Decoder ffwd {i}: {np.linalg.norm(self.decoder[i].ffwd.dinputs)}')
            print(f'Decoder cross MHA {i}: {np.linalg.norm(self.decoder[i].cross_mha.dinputs_query)} | {np.linalg.norm(self.decoder[i].cross_mha.dinputs_context)}')
            print(f'Decoder mask MHA {i}: {np.linalg.norm(self.decoder[i].mask_mha.dinputs)}')
            print(f'Decoder grad for encoder {i}: {np.linalg.norm(self.decoder[i].grad_wrt_encoder_output)}')
        for i in range(len(self.encoder)):
            print(f'Encoder ffwd {i}: {np.linalg.norm(self.encoder[i].ffwd.dinputs)}')
            print(f'Encoder MHA {i}: {np.linalg.norm(self.encoder[i].mha.dinputs_query)} | {np.linalg.norm(self.encoder[i].mha.dinputs_context)}')
            print(f'Encoder dipnuts {i}: {np.linalg.norm(self.encoder[i].dinputs)}')
        print(f'------------------------------------------')

    def train(self, X, y_dec, y_true, epochs = 1000, batch_size: int = None, print_every: int = None):

        self.loss_vals = []
        self.i_vals = []

        for i in range(epochs + 1):
            #batch_X, batch_y_dec, batch_y_true = get_random_batch(X, y_dec, y_true, batch_size)
            #batch_X, batch_y_dec, batch_y_true = batch_X.astype(int), batch_y_dec.astype(int), batch_y_true.astype(int)

            self.forward(X, y_dec, training=True)
            
            #print(f'output: {self.output}, y: {y_true}')
            self.backward(self.output, y_true)

            self.optimizer.pre_update_parameters()
            self.optimizer.update_parameters(self)
            self.optimizer.post_update_parameters()

            if print_every is not None and not i % print_every:
                output = self.output
                B, T, C = output.shape
                print(f'===== EPOCH : {i} ===== LOSS : {self.loss_function.calculate(output.reshape(B*T, C), y_true.reshape(B*T))} =====')
                self.loss_vals.append(self.loss_function.calculate(output.reshape(B*T, C), y_true.reshape(B*T)))
                self.i_vals.append(i)
                #self._print_grad_norms()
                

    def generate(self, inputs_enc, max_len=5, start_token=10, end_token=11):
        """
        Generate output sequence given input encoding.
        
        inputs_enc: np.ndarray of shape (batch_size, source_seq_len)
        max_len: max length of generated sequence
        start_token: int, index of <SOS> token
        end_token: int, index of <EOS> token
        
        Returns:
            generated sequences of shape (batch_size, generated_seq_len)
        """
        batch_size = inputs_enc.shape[0]
        
        # Run encoder once
        self.embed_enc.forward(inputs_enc, training=False)
        self.pos_enc_enc.forward(self.embed_enc.output, training=False)
        self.encoder[0].forward(self.pos_enc_enc.output, training=False)
        for idx, enc in enumerate(self.encoder[1:], start=1):
            enc.forward(self.encoder[idx-1].output, training=False)
        encoder_output = self.encoder[-1].output
        
        # Initialize decoder input with start tokens (shape: batch_size x 1)
        decoder_input = np.full((batch_size, 1), start_token, dtype=int)
        
        generated = decoder_input.copy()
        
        for _ in range(max_len):
            # Decoder forward pass for current decoder input
            self.embed_dec.forward(decoder_input, training=False)
            self.pos_enc_dec.forward(self.embed_dec.output, training=False)

            self.decoder[0].forward(self.pos_enc_dec.output, self.encoder[-1].output, training=False)
            for idx, dec in enumerate(self.decoder[1:], start=1):
                dec.forward(self.decoder[idx-1].output, self.encoder[-1].output, training=False)

            self.linear.forward(self.decoder[-1].output)
            self.softm.forward(self.linear.output)
            
            # Get last timestep prediction probs (batch_size, vocab_size)
            probs = self.softm.output[:, -1, :]
            
            # Greedy decode: pick the highest probability token for each example
            next_tokens = np.argmax(probs, axis=1).reshape(-1, 1)
            
            # Append to generated sequences
            generated = np.concatenate([generated, next_tokens], axis=1)
            
            # Prepare next decoder input
            decoder_input = generated
            
            # Stop if all sequences generated <EOS>
            if np.all(next_tokens == end_token):
                break
        
        return generated